## Imports

In [1]:
import os
import sys
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## Load split data

In [2]:
X_train = pd.read_csv("../data/processed/X_train.csv")
X_dev = pd.read_csv("../data/processed/X_dev.csv")
X_test = pd.read_csv("../data/processed/X_test.csv")

y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()
y_dev = pd.read_csv("../data/processed/y_dev.csv").squeeze()
y_test = pd.read_csv("../data/processed/y_test.csv").squeeze()

print("Train:", X_train.shape)
print("Dev:", X_dev.shape)
print("Test:", X_test.shape)

Train: (60996, 29)
Dev: (13071, 29)
Test: (13071, 29)


## Feature engineering transformer

In [5]:
class HotelFeatureEngineer(BaseEstimator, TransformerMixin):

    def __init__(self, include_room_change=True):
        self.include_room_change = include_room_change

    def fit(self, X, y=None):
        X = X.copy()

        # Learn ADR cap ONLY from training data
        adr_values = pd.to_numeric(X["adr"], errors="coerce")

        if adr_values.notna().any():
            self.adr_upper_ = adr_values[adr_values >= 0].quantile(0.995)
        else:
            self.adr_upper_ = 0.0

        return self

    def transform(self, X):
        X = X.copy()

        # Missing value handling

        if "children" in X.columns:
            X["children"] = X["children"].fillna(0)
            X["children"] = X["children"].clip(lower=0, upper=3)

        if "country" in X.columns:
            X["country"] = X["country"].fillna("Unknown")

        if "agent" in X.columns:
            X["agent"] = X["agent"].fillna("No Agent")

        if "company" in X.columns:
            X["company"] = X["company"].fillna("No Company")

        # ADR cleaning

        X["adr"] = pd.to_numeric(X["adr"], errors="coerce")

        X.loc[X["adr"] < 0, "adr"] = 0

        X["adr"] = X["adr"].fillna(0)

        # Cap using training-derived threshold
        X["adr"] = X["adr"].clip(upper=self.adr_upper_)


        # Guest count validation

        X["total_guests"] = (
            X["adults"] +
            X["children"] +
            X["babies"]
        )

        X["is_invalid_guest_count"] = (
            X["total_guests"] <= 0
        ).astype(int)


        # Total nights


        X["total_nights"] = (
            X["stays_in_weekend_nights"] +
            X["stays_in_week_nights"]
        )
        # Weekend night ratio

        X["weekend_night_ratio"] = np.where(
            X["total_nights"] > 0,
            X["stays_in_weekend_nights"] / X["total_nights"],
            0
        )

        # Previous booking behaviour

        X["total_previous_bookings"] = (
            X["previous_cancellations"] +
            X["previous_bookings_not_canceled"]
        )

        X["prior_cancellation_rate"] = np.where(
            X["total_previous_bookings"] > 0,
            X["previous_cancellations"] /
            X["total_previous_bookings"],
            0
        )
        # Family indicator

        X["is_family"] = (
            (X["children"] > 0) |
            (X["babies"] > 0)
        ).astype(int)

        # Arrival season
        season_map = {
            "December": "Winter",
            "January": "Winter",
            "February": "Winter",

            "March": "Spring",
            "April": "Spring",
            "May": "Spring",

            "June": "Summer",
            "July": "Summer",
            "August": "Summer",

            "September": "Autumn",
            "October": "Autumn",
            "November": "Autumn"
        }

        X["arrival_season"] = (
            X["arrival_date_month"]
            .map(season_map)
            .fillna("Unknown")
        )

        # Original month is not required after season engineering
        X = X.drop(columns=["arrival_date_month"], errors="ignore")

        # Room type change


        if self.include_room_change:

            X["room_type_changed"] = (
                X["reserved_room_type"] != X["assigned_room_type"]
            ).astype(int)

        return X

| # | New Feature               | What it represents                                                 |
| - | ------------------------- | ------------------------------------------------------------------ |
| 1 | `total_guests`            | Total number of guests = adults + children + babies                |
| 2 | `is_invalid_guest_count`  | Flags bookings with 0 or fewer guests                              |
| 3 | `total_nights`            | Total stay length = weekend nights + weekday nights                |
| 4 | `weekend_night_ratio`     | Proportion of the stay spent on weekend nights                     |
| 5 | `total_previous_bookings` | Total previous bookings = canceled + non-canceled                  |
| 6 | `prior_cancellation_rate` | Percentage of previous bookings that were canceled                 |
| 7 | `is_family`               | Indicates whether children or babies are included                  |
| 8 | `arrival_season`          | Converts arrival month into Winter, Spring, Summer, or Autumn      |
| 9 | `room_type_changed`       | Indicates whether the assigned room differs from the reserved room |


## Top-N categorical bucketing

In [6]:
class TopNBucketer(BaseEstimator, TransformerMixin):

    def __init__(self, top_n=10):
        self.top_n = top_n

    def fit(self, X, y=None):

        X = pd.DataFrame(X)

        self.top_categories_ = []

        for column in X.columns:

            counts = X[column].value_counts(dropna=False)

            top_values = list(
                counts.head(self.top_n).index
            )

            self.top_categories_.append(set(top_values))

        return self

    def transform(self, X):

        X = pd.DataFrame(X).copy()

        for i, column in enumerate(X.columns):

            allowed = self.top_categories_[i]

            X[column] = X[column].where(
                X[column].isin(allowed),
                "Other"
            )

        return X.to_numpy()

## Define preprocessing columns

In [7]:
LOW_CARDINALITY_CATEGORICAL = [
    "hotel",
    "meal",
    "market_segment",
    "distribution_channel",
    "deposit_type",
    "customer_type",
    "arrival_season",
    "reserved_room_type"
]

FULL_LIFECYCLE_CATEGORICAL = [
    "assigned_room_type"
]

HIGH_CARDINALITY_CATEGORICAL = [
    "country",
    "agent",
    "company"
]

NUMERIC_COLUMNS = [
    "lead_time",
    "arrival_date_year",
    "arrival_date_week_number",
    "arrival_date_day_of_month",
    "stays_in_weekend_nights",
    "stays_in_week_nights",
    "adults",
    "children",
    "babies",
    "is_repeated_guest",
    "previous_cancellations",
    "previous_bookings_not_canceled",
    "booking_changes",
    "days_in_waiting_list",
    "adr",
    "required_car_parking_spaces",
    "total_of_special_requests",
    "total_nights",
    "weekend_night_ratio",
    "total_guests",
    "total_previous_bookings",
    "prior_cancellation_rate",
    "is_invalid_guest_count",
    "is_family"
]

## Create preprocessing function

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer


def make_preprocessor(
    scale_numeric=False,
    day_zero=False
):

    # Numeric features

    numeric_columns = NUMERIC_COLUMNS.copy()

    if day_zero:
        numeric_columns.remove("booking_changes")
        numeric_columns.remove("days_in_waiting_list")

    if scale_numeric:

        numeric_pipeline = Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="median")
            ),
            (
                "scaler",
                StandardScaler()
            )
        ])

    else:

        numeric_pipeline = Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="median")
            )
        ])

    # Low-cardinality categories
    categorical_columns = LOW_CARDINALITY_CATEGORICAL.copy()

    if not day_zero:
        categorical_columns += FULL_LIFECYCLE_CATEGORICAL

    categorical_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="Unknown"
            )
        ),
        (
            "to_string",
            FunctionTransformer(
                lambda X: X.astype(str)
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            )
        )
    ])

    # High-cardinality categories

    high_cardinality_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="Unknown"
            )
        ),
        (
            "to_string",
            FunctionTransformer(
                lambda X: X.astype(str)
            )
        ),
        (
            "top_n",
            TopNBucketer(top_n=10)
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            )
        )
    ])

    # Combine all transformers

    transformers = [
        (
            "numeric",
            numeric_pipeline,
            numeric_columns
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_columns
        ),
        (
            "high_cardinality",
            high_cardinality_pipeline,
            HIGH_CARDINALITY_CATEGORICAL
        )
    ]

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop"
    )

## Create complete model pipeline

In [12]:
def make_model_pipeline(
    model,
    scale_numeric=False,
    day_zero=False
):

    pipeline = Pipeline([
        (
            "features",
            HotelFeatureEngineer(
                include_room_change=not day_zero
            )
        ),
        (
            "preprocessing",
            make_preprocessor(
                scale_numeric=scale_numeric,
                day_zero=day_zero
            )
        ),
        (
            "model",
            model
        )
    ])

    return pipeline

## Test the preprocessing pipeline

In [13]:
from sklearn.tree import DecisionTreeClassifier

test_model = DecisionTreeClassifier(
    random_state=42,
    class_weight="balanced"
)

test_pipeline = make_model_pipeline(
    model=test_model,
    scale_numeric=False,
    day_zero=False
)

test_pipeline.fit(X_train, y_train)

X_train_transformed = test_pipeline.named_steps[
    "preprocessing"
].transform(
    test_pipeline.named_steps[
        "features"
    ].transform(X_train)
)

print("Transformed shape:", X_train_transformed.shape)
print("Preprocessing pipeline test passed.")

Transformed shape: (60996, 109)
Preprocessing pipeline test passed.


## Verify no NaN / infinite values

In [14]:
from scipy import sparse

if sparse.issparse(X_train_transformed):

    assert np.isfinite(
        X_train_transformed.data
    ).all()

else:

    assert np.isfinite(
        X_train_transformed
    ).all()

print("No NaN or infinite values found after preprocessing.")

No NaN or infinite values found after preprocessing.


In [18]:
# ==========================================
# Preprocessing Validation
# ==========================================

# 1. Apply feature engineering
X_train_features = test_pipeline.named_steps[
    "features"
].transform(X_train)

print("Feature-engineered shape:", X_train_features.shape)

# 2. Display new features
new_features = [
    "total_guests",
    "is_invalid_guest_count",
    "total_nights",
    "weekend_night_ratio",
    "total_previous_bookings",
    "prior_cancellation_rate",
    "is_family",
    "arrival_season",
    "room_type_changed"
]

print("\nNew features created:")
print(new_features)

# 3. Check that new features exist
missing_features = [
    feature for feature in new_features
    if feature not in X_train_features.columns
]

assert len(missing_features) == 0, \
    f"Missing features: {missing_features}"

print("\nAll new features are present.")

# 4. Display sample
print("\nSample feature-engineered data:")
display(X_train_features.head())

# 5. Apply preprocessing
X_train_processed = test_pipeline.named_steps[
    "preprocessing"
].transform(X_train_features)

print("\nPreprocessed shape:", X_train_processed.shape)


print(
    "\nNumber of processed features:",
    X_train_processed.shape[1]
)

# 7. Check for NaN values
if hasattr(X_train_processed, "data"):
    has_nan = np.isnan(X_train_processed.data).any()
else:
    has_nan = np.isnan(X_train_processed).any()

print("\nAny NaN values:", has_nan)

assert has_nan == False, \
    "ERROR: NaN values are present after preprocessing."

print("\nPreprocessing validation completed successfully.")

Feature-engineered shape: (60996, 37)

New features created:
['total_guests', 'is_invalid_guest_count', 'total_nights', 'weekend_night_ratio', 'total_previous_bookings', 'prior_cancellation_rate', 'is_family', 'arrival_season', 'room_type_changed']

All new features are present.

Sample feature-engineered data:


,hotel,lead_time,arrival_date_year,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,...,total_of_special_requests,total_guests,is_invalid_guest_count,total_nights,weekend_night_ratio,total_previous_bookings,prior_cancellation_rate,is_family,arrival_season,room_type_changed
0,City Hotel,43,2017,1,7,2,3,2,0.0,0,...,0,2.0,0,5,0.40,0,0.0,0,Winter,0
1,City Hotel,174,2017,21,23,0,3,2,0.0,0,...,1,2.0,0,3,0.00,0,0.0,0,Spring,0
2,Resort Hotel,0,2017,4,25,0,2,1,0.0,0,...,0,1.0,0,2,0.00,2,0.0,0,Winter,0
3,City Hotel,16,2016,10,3,1,3,2,0.0,0,...,1,2.0,0,4,0.25,0,0.0,0,Spring,0
4,City Hotel,92,2015,52,26,2,3,2,0.0,0,...,1,2.0,0,5,0.40,0,0.0,0,Winter,0



Preprocessed shape: (60996, 109)

Number of processed features: 109

Any NaN values: False

Preprocessing validation completed successfully.
